In [ ]:
from ultralytics import YOLO
from pathlib import Path

BASE_DIR = Path.cwd()
PRETRAINED_DIR = BASE_DIR.parent / "models" / "pretrained"
RUNS_DIR = BASE_DIR.parent / "runs" / "detect"
DATA_CONFIG = BASE_DIR.parent / "datasets" / "pcb-defects-final" / "data.yaml"

model = YOLO(str(PRETRAINED_DIR / 'yolo11m.pt'))

model.train(
    data=str(DATA_CONFIG),
    project=str(RUNS_DIR),
    epochs=100,
    imgsz=640,
    batch=12,
    device=0,
    
    # Оптимизатор и скорость обучения
    optimizer="AdamW",
    lr0=1e-4,                   # Начальный learning rate
    lrf=0.01,                   # Конечный LR (коэффициент от lr0 для Cosine Annealing)
    warmup_epochs=5.0,
    momentum=0.9,               # beta1
    bgr=0.999,                  # beta2 (в Ultralytics реализуется через доп. параметры)
    weight_decay=5e-4,
    amp=True,
    
    # Функции потерь (Weight Coefficients)
    # λ_cls = 0.5, λ_box (loc) = 7.5, λ_dfl (аналог obj в новых YOLO) = 1.0
    cls=0.5,                    
    box=7.5,                    
    dfl=1.0,                    
    
    # Аугментации
    mosaic=0.8,
    mixup=0.15,
    scale=0.5,                  # Random Scaling (0.5-1.5x)
    hsv_h=0.015,                # HSV-Hue
    hsv_s=0.7,                  # HSV-Saturation
    hsv_v=0.4,                  # HSV-Value
    flipud=0.0,                 
    fliplr=0.5,                 # Random Flip (p=0.5)
    degrees=10.0,               # Random Rotation (+/- 10)
    
    # Параметры валидации/NMS
    iou=0.5,                    # NMS IoU threshold
    conf=0.25,                   # Confidence threshold

    name='pcb_defect_detector_v14_yolov11m'
)

In [ ]:
from ultralytics import RTDETR
from pathlib import Path

BASE_DIR = Path.cwd()
PRETRAINED_DIR = BASE_DIR.parent / "models" / "pretrained"
RUNS_DIR = BASE_DIR.parent / "runs" / "detect"
DATA_CONFIG = BASE_DIR.parent / "datasets" / "pcb-defects-final" / "data.yaml"

model = RTDETR(str(PRETRAINED_DIR /'rtdetr-l.pt'))

model.train(
    data=str(DATA_CONFIG),
    project=str(RUNS_DIR),
    epochs=100,
    imgsz=640,
    batch=8,
    lr0=1e-4,
    optimizer='AdamW',
    warmup_epochs=5,
    amp=True,
    mosaic=0.8,
    mixup=0.15,
    name='pcb_defect_detector_v14_rtdetrv2m'
)

In [ ]:
from ultralytics import RTDETR
from pathlib import Path

BASE_DIR = Path.cwd()
RUNS_DIR = BASE_DIR.parent / "runs" / "detect"
DATA_CONFIG = BASE_DIR.parent / "datasets" / "pcb-defects-final" / "data.yaml"

model = RTDETR('../runs/detect/pcb_defect_detector_v14_rtdetrv2m/weights/last.pt')

model.train(
    data=str(DATA_CONFIG),
    project=str(RUNS_DIR),
    epochs=62,
    imgsz=640,
    batch=8,
    lr0=1e-5,
    name='pcb_defect_detector_v14_rtdetrv2m_continued'
)

In [ ]:
import torch
import segmentation_models_pytorch as smp
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

# Инициализация архитектуры
model = smp.Unet(
    encoder_name="resnet34",        # Backbone
    encoder_weights=None,           # Обучение с нуля
    in_channels=3,
    classes=1                       # Бинарная сегментация
)

# Настройка оптимизатора
optimizer = Adam(model.parameters(), lr=5e-4, betas=(0.9, 0.999), weight_decay=1e-4)

# Планировщик
scheduler = CosineAnnealingLR(optimizer, T_max=100)

# Комбинированная функция потерь (BCE + Dice 1:1)
dice_loss = smp.losses.DiceLoss(mode='binary')
bce_loss = torch.nn.BCEWithLogitsLoss()

def criterion(y_pred, y_true):
    return 0.5 * bce_loss(y_pred, y_true) + 0.5 * dice_loss(y_pred, y_true)

# В цикле обучения:
# - Использовать torch.cuda.amp.autocast() для Mixed Precision
# - Batch size = 8

In [ ]:
import numpy as np
import cv2
from torch.utils.data import Dataset
import os
import segmentation_models_pytorch as smp
import torch
from torch.amp import GradScaler, autocast
from tqdm import tqdm

def create_mask_from_yolo(label_path, img_size=(512, 512)):
    mask = np.zeros(img_size, dtype=np.float32)
    if not os.path.exists(label_path): # Проверка на случай отсутствия дефектов
        return mask
        
    with open(label_path, 'r') as f:
        for line in f.readlines():
            parts = line.split()
            if len(parts) < 5: continue
            cls, x, y, w, h = map(float, parts)
            
            # Масштабируем под текущий img_size
            x1 = int((x - w/2) * img_size[1])
            y1 = int((y - h/2) * img_size[0])
            x2 = int((x + w/2) * img_size[1])
            y2 = int((y + h/2) * img_size[0])
            
            # Ограничиваем координаты, чтобы не выйти за рамки массива
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(img_size[1], x2), min(img_size[0], y2)
            
            mask[y1:y2, x1:x2] = 1.0
    return mask

def get_dice_score(output, target, smooth=1e-6):
    # Превращаем логиты модели в бинарную маску (0 или 1)
    output = torch.sigmoid(output)
    output = (output > 0.5).float()
    
    intersection = (output * target).sum()
    return (2. * intersection + smooth) / (output.sum() + target.sum() + smooth)

class PCBDataset(Dataset):
    def __init__(self, img_dir, label_dir, transform=None):
        self.img_names = os.listdir(img_dir)
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transform = transform

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_names[idx])
        label_path = os.path.join(self.label_dir, self.img_names[idx].replace('.jpg', '.txt'))
        
        # 1. Загрузка
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # 2. МАСШТАБИРОВАНИЕ (Resize)
        target_size = (640, 640)
        image = cv2.resize(image, target_size)
        mask = create_mask_from_yolo(label_path, img_size=target_size)

        # 3. Транспонирование и тензоры
        image = image.transpose(2, 0, 1) 
        image = torch.from_numpy(image).float() / 255.0
        mask = torch.from_numpy(mask).float()

        return image, mask
    
# Архитектура
model = smp.Unet(
    encoder_name="resnet34", 
    encoder_weights=None, # Обучение с нуля
    in_channels=3, 
    classes=1
).to('cuda')

# Параметры
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)
criterion_dice = smp.losses.DiceLoss(mode='binary', from_logits=True)
criterion_bce = torch.nn.BCEWithLogitsLoss()

scaler = GradScaler()

my_data = PCBDataset(img_dir='../datasets/pcb-defects-final/train/images', label_dir='../datasets/pcb-defects-final/train/labels')

train_loader = torch.utils.data.DataLoader(my_data, batch_size=8, shuffle=True)

save_dir = '../runs/detect/u_net_resnet34_weights'
os.makedirs(save_dir, exist_ok=True)

best_dice = 0.0

for epoch in range(100):
    model.train()
    train_loss = 0.0
    train_dice = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/100")

    for images, masks in pbar:
        images, masks = images.to('cuda'), masks.to('cuda').unsqueeze(1)
        
        optimizer.zero_grad()
        
        with autocast(device_type="cuda"): # Mixed Precision
            outputs = model(images)
            loss = 0.5 * criterion_dice(outputs, masks) + 0.5 * criterion_bce(outputs, masks)
            # Считаем Dice прямо во время обучения
            current_dice = get_dice_score(outputs.detach(), masks)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        train_dice += current_dice.item()
        
        # Обновляем текст в консоли
        pbar.set_postfix({
            'loss': f"{loss.item():.4f}", 
            'dice': f"{current_dice.item():.4f}"
        })
    
    # Считаем средние показатели за эпоху
    avg_loss = train_loss / len(train_loader)
    avg_dice = train_dice / len(train_loader)

    # Сохранение после каждой эпохи
    torch.save(model.state_dict(), os.path.join(save_dir, 'last.pth'))
    
    # Логика сохранения ЛУЧШЕЙ модели
    if avg_dice > best_dice:
        best_dice = avg_dice
        torch.save(model.state_dict(), os.path.join(save_dir, 'best.pth'))
        print(f"--- New Best Model Saved (Dice: {best_dice:.4f}) ---")

    # Вывод итогов эпохи
    print(f"Epoch {epoch+1} Summary: Avg Loss: {avg_loss:.4f}, Avg Dice: {avg_dice:.4f}")
    
    # Опционально: записываем логи в текстовый файл
    with open(os.path.join(save_dir, 'log.txt'), 'a') as f:
        f.write(f"Epoch {epoch+1}: loss={avg_loss:.4f}, dice={avg_dice:.4f}\n")
        
    print(f"Epoch {epoch+1} completed")